In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import uuid

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customer360_dev.control.pipeline_audit
(
    audit_run_id STRING,
    pipeline_run_id STRING,

    entity_name STRING,
    pipeline_name STRING,
    layer STRING,

    source_system STRING,
    source_object STRING,
    target_object STRING,

    load_type STRING,

    run_status STRING,

    start_timestamp TIMESTAMP,
    end_timestamp TIMESTAMP,

    input_row_count BIGINT,
    output_row_count BIGINT,
    rejected_row_count BIGINT,

    insert_count BIGINT,
    update_count BIGINT,
    unchanged_count BIGINT,

    reconciliation_status STRING,

    error_class STRING,
    error_message STRING,

    notebook_name STRING,

    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customer360_dev.control.reconciliation_results
(
    reconciliation_id STRING,
    pipeline_run_id STRING,

    entity_name STRING,
    layer STRING,

    source_count BIGINT,
    target_count BIGINT,
    rejected_count BIGINT,
    duplicate_count BIGINT,

    difference_count BIGINT,

    reconciliation_status STRING,

    reconciliation_timestamp TIMESTAMP,

    details STRING,

    created_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customer360_dev.control.data_quality_results
(
    dq_result_id STRING,
    pipeline_run_id STRING,

    entity_name STRING,
    layer STRING,

    rule_name STRING,
    rule_description STRING,

    severity STRING,

    checked_row_count BIGINT,
    failed_row_count BIGINT,

    dq_status STRING,

    check_timestamp TIMESTAMP,

    created_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customer360_dev.control.pipeline_config
(
    entity_name STRING,

    source_system STRING,
    source_path STRING,

    bronze_table STRING,
    silver_table STRING,
    gold_table STRING,

    business_key STRING,

    load_enabled BOOLEAN,
    cdc_enabled BOOLEAN,

    load_sequence INT,

    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
pipeline_config_data = [

    (
        "customers",
        "CRM",
        "/Volumes/customer360_dev/staging/source_files/customers/",
        "customer360_dev.bronze.customers",
        "customer360_dev.silver.customers",
        "customer360_dev.gold.dim_customer",
        "customer_id",
        True,
        True,
        1
    ),

    (
        "addresses",
        "CRM",
        "/Volumes/customer360_dev/staging/source_files/addresses/",
        "customer360_dev.bronze.addresses",
        "customer360_dev.silver.addresses",
        None,
        "address_id",
        True,
        True,
        2
    ),

    (
        "subscriptions",
        "SUBSCRIPTION_PLATFORM",
        "/Volumes/customer360_dev/staging/source_files/subscriptions/",
        "customer360_dev.bronze.subscriptions",
        "customer360_dev.silver.subscriptions",
        None,
        "subscription_id",
        True,
        True,
        3
    ),

    (
        "products",
        "PRODUCT_CATALOG",
        "/Volumes/customer360_dev/staging/source_files/products/",
        "customer360_dev.bronze.products",
        "customer360_dev.silver.products",
        "customer360_dev.gold.dim_product",
        "product_id",
        True,
        True,
        4
    ),

    (
        "orders",
        "ORDER_MANAGEMENT",
        "/Volumes/customer360_dev/staging/source_files/orders/",
        "customer360_dev.bronze.orders",
        "customer360_dev.silver.orders",
        "customer360_dev.gold.fact_orders",
        "order_id",
        True,
        True,
        5
    ),

    (
        "order_items",
        "ORDER_MANAGEMENT",
        "/Volumes/customer360_dev/staging/source_files/order_items/",
        "customer360_dev.bronze.order_items",
        "customer360_dev.silver.order_items",
        "customer360_dev.gold.fact_order_items",
        "order_item_id",
        True,
        True,
        6
    ),

    (
        "payments",
        "PAYMENT_PLATFORM",
        "/Volumes/customer360_dev/staging/source_files/payments/",
        "customer360_dev.bronze.payments",
        "customer360_dev.silver.payments",
        "customer360_dev.gold.fact_payments",
        "payment_id",
        True,
        True,
        7
    ),

    (
        "support_tickets",
        "SUPPORT_PLATFORM",
        "/Volumes/customer360_dev/staging/source_files/support_tickets/",
        "customer360_dev.bronze.support_tickets",
        "customer360_dev.silver.support_tickets",
        "customer360_dev.gold.fact_support_tickets",
        "ticket_id",
        True,
        True,
        8
    )
]

In [0]:
config_df = spark.createDataFrame(
    pipeline_config_data,
    """
    entity_name STRING,
    source_system STRING,
    source_path STRING,
    bronze_table STRING,
    silver_table STRING,
    gold_table STRING,
    business_key STRING,
    load_enabled BOOLEAN,
    cdc_enabled BOOLEAN,
    load_sequence INT
    """
)

config_df = (
    config_df
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

In [0]:
(
    config_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "customer360_dev.control.pipeline_config"
    )
)

print("Pipeline configuration created.")

Pipeline configuration created.


In [0]:
def start_pipeline_audit(
    pipeline_run_id,
    entity_name,
    pipeline_name,
    layer,
    source_system,
    source_object,
    target_object,
    load_type,
    notebook_name
):

    audit_run_id = str(uuid.uuid4())

    audit_df = (
        spark.createDataFrame(
            [
                (
                    audit_run_id,
                    pipeline_run_id,
                    entity_name,
                    pipeline_name,
                    layer,
                    source_system,
                    source_object,
                    target_object,
                    load_type,
                    "RUNNING",
                    notebook_name
                )
            ],
            """
            audit_run_id STRING,
            pipeline_run_id STRING,
            entity_name STRING,
            pipeline_name STRING,
            layer STRING,
            source_system STRING,
            source_object STRING,
            target_object STRING,
            load_type STRING,
            run_status STRING,
            notebook_name STRING
            """
        )

        .withColumn(
            "start_timestamp",
            F.current_timestamp()
        )

        .withColumn(
            "end_timestamp",
            F.lit(None).cast("timestamp")
        )

        .withColumn(
            "input_row_count",
            F.lit(None).cast("long")
        )

        .withColumn(
            "output_row_count",
            F.lit(None).cast("long")
        )

        .withColumn(
            "rejected_row_count",
            F.lit(None).cast("long")
        )

        .withColumn(
            "insert_count",
            F.lit(None).cast("long")
        )

        .withColumn(
            "update_count",
            F.lit(None).cast("long")
        )

        .withColumn(
            "unchanged_count",
            F.lit(None).cast("long")
        )

        .withColumn(
            "reconciliation_status",
            F.lit(None).cast("string")
        )

        .withColumn(
            "error_class",
            F.lit(None).cast("string")
        )

        .withColumn(
            "error_message",
            F.lit(None).cast("string")
        )

        .withColumn(
            "created_at",
            F.current_timestamp()
        )

        .withColumn(
            "updated_at",
            F.current_timestamp()
        )
    )

    (
        audit_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "customer360_dev.control.pipeline_audit"
        )
    )

    print(
        "Audit started:",
        audit_run_id
    )

    return audit_run_id

In [0]:
def complete_pipeline_audit(
    audit_run_id,
    input_row_count,
    output_row_count,
    rejected_row_count=0,
    insert_count=0,
    update_count=0,
    unchanged_count=0,
    reconciliation_status="PASS"
):

    spark.sql(
        f"""
        UPDATE customer360_dev.control.pipeline_audit

        SET
            run_status = 'SUCCESS',

            end_timestamp = current_timestamp(),

            input_row_count = {input_row_count},
            output_row_count = {output_row_count},
            rejected_row_count = {rejected_row_count},

            insert_count = {insert_count},
            update_count = {update_count},
            unchanged_count = {unchanged_count},

            reconciliation_status =
                '{reconciliation_status}',

            updated_at = current_timestamp()

        WHERE audit_run_id = '{audit_run_id}'
        """
    )

    print(
        "Audit completed successfully:",
        audit_run_id
    )

In [0]:
def fail_pipeline_audit(
    audit_run_id,
    error_class,
    error_message
):

    safe_error_class = (
        str(error_class)
        .replace("'", "''")
    )

    safe_error_message = (
        str(error_message)
        .replace("'", "''")
    )

    spark.sql(
        f"""
        UPDATE customer360_dev.control.pipeline_audit

        SET
            run_status = 'FAILED',

            end_timestamp =
                current_timestamp(),

            error_class =
                '{safe_error_class}',

            error_message =
                '{safe_error_message}',

            updated_at =
                current_timestamp()

        WHERE audit_run_id =
            '{audit_run_id}'
        """
    )

    print(
        "Audit marked FAILED:",
        audit_run_id
    )

In [0]:
def log_reconciliation(
    pipeline_run_id,
    entity_name,
    layer,
    source_count,
    target_count,
    rejected_count,
    duplicate_count,
    reconciliation_status,
    details=None
):

    reconciliation_id = str(uuid.uuid4())

    difference_count = (
        source_count
        -
        target_count
        -
        rejected_count
    )

    reconciliation_df = (
        spark.createDataFrame(
            [
                (
                    reconciliation_id,
                    pipeline_run_id,
                    entity_name,
                    layer,
                    int(source_count),
                    int(target_count),
                    int(rejected_count),
                    int(duplicate_count),
                    int(difference_count),
                    reconciliation_status,
                    details
                )
            ],
            """
            reconciliation_id STRING,
            pipeline_run_id STRING,
            entity_name STRING,
            layer STRING,
            source_count LONG,
            target_count LONG,
            rejected_count LONG,
            duplicate_count LONG,
            difference_count LONG,
            reconciliation_status STRING,
            details STRING
            """
        )

        .withColumn(
            "reconciliation_timestamp",
            F.current_timestamp()
        )

        .withColumn(
            "created_at",
            F.current_timestamp()
        )
    )

    (
        reconciliation_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "customer360_dev.control.reconciliation_results"
        )
    )

In [0]:
def log_dq_result(
    pipeline_run_id,
    entity_name,
    layer,
    rule_name,
    rule_description,
    severity,
    checked_row_count,
    failed_row_count
):

    dq_result_id = str(uuid.uuid4())

    dq_status = (
        "PASS"
        if failed_row_count == 0
        else "FAIL"
    )

    dq_df = (
        spark.createDataFrame(
            [
                (
                    dq_result_id,
                    pipeline_run_id,
                    entity_name,
                    layer,
                    rule_name,
                    rule_description,
                    severity,
                    int(checked_row_count),
                    int(failed_row_count),
                    dq_status
                )
            ],
            """
            dq_result_id STRING,
            pipeline_run_id STRING,
            entity_name STRING,
            layer STRING,
            rule_name STRING,
            rule_description STRING,
            severity STRING,
            checked_row_count LONG,
            failed_row_count LONG,
            dq_status STRING
            """
        )

        .withColumn(
            "check_timestamp",
            F.current_timestamp()
        )

        .withColumn(
            "created_at",
            F.current_timestamp()
        )
    )

    (
        dq_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "customer360_dev.control.data_quality_results"
        )
    )

In [0]:
pipeline_run_id = str(uuid.uuid4())

audit_run_id = start_pipeline_audit(

    pipeline_run_id=pipeline_run_id,

    entity_name="customers",

    pipeline_name="customer_incremental_cdc",

    layer="BRONZE_TO_SILVER",

    source_system="CRM",

    source_object=
        "customer360_dev.bronze.customers",

    target_object=
        "customer360_dev.silver.customers",

    load_type="INCREMENTAL",

    notebook_name=
        "customer_incremental_pipeline"
)

Audit started: 2d28eb7f-6dfe-4f25-80d9-b15ea2a9a484


In [0]:
complete_pipeline_audit(

    audit_run_id=audit_run_id,

    input_row_count=1000,

    output_row_count=995,

    rejected_row_count=5,

    insert_count=100,

    update_count=200,

    unchanged_count=695,

    reconciliation_status="PASS"
)

Audit completed successfully: 2d28eb7f-6dfe-4f25-80d9-b15ea2a9a484


In [0]:
log_reconciliation(

    pipeline_run_id=pipeline_run_id,

    entity_name="customers",

    layer="BRONZE_TO_SILVER",

    source_count=1000,

    target_count=995,

    rejected_count=5,

    duplicate_count=0,

    reconciliation_status="PASS",

    details="Test audit-framework reconciliation record"
)

In [0]:
log_dq_result(

    pipeline_run_id=pipeline_run_id,

    entity_name="customers",

    layer="SILVER",

    rule_name="VALID_EMAIL",

    rule_description=
        "Customer email must match expected email format",

    severity="CRITICAL",

    checked_row_count=1000,

    failed_row_count=0
)

In [0]:
%sql
SELECT
    pipeline_run_id,
    entity_name,
    pipeline_name,
    layer,
    load_type,
    run_status,
    input_row_count,
    output_row_count,
    rejected_row_count,
    reconciliation_status,
    start_timestamp,
    end_timestamp
FROM customer360_dev.control.pipeline_audit
ORDER BY created_at DESC;

pipeline_run_id,entity_name,pipeline_name,layer,load_type,run_status,input_row_count,output_row_count,rejected_row_count,reconciliation_status,start_timestamp,end_timestamp
d59e7970-07dd-4656-a2c7-71b2b40c90a5,customers,customer_incremental_cdc,BRONZE_TO_SILVER,INCREMENTAL,SUCCESS,1000,995,5,PASS,2026-09-29T23:50:11.637Z,2026-09-29T23:50:22.742Z


In [0]:
%sql
SELECT *
FROM customer360_dev.control.reconciliation_results
ORDER BY created_at DESC;

reconciliation_id,pipeline_run_id,entity_name,layer,source_count,target_count,rejected_count,duplicate_count,difference_count,reconciliation_status,reconciliation_timestamp,details,created_at
ea018a3c-539a-4afd-ba7b-7642530aabe4,d59e7970-07dd-4656-a2c7-71b2b40c90a5,customers,BRONZE_TO_SILVER,1000,995,5,0,0,PASS,2026-09-29T23:50:31.413Z,Test audit-framework reconciliation record,2026-09-29T23:50:31.413Z


In [0]:
%sql
SELECT *
FROM customer360_dev.control.data_quality_results
ORDER BY created_at DESC;

dq_result_id,pipeline_run_id,entity_name,layer,rule_name,rule_description,severity,checked_row_count,failed_row_count,dq_status,check_timestamp,created_at
850cccd7-7998-44e9-a22a-e9298eb70732,d59e7970-07dd-4656-a2c7-71b2b40c90a5,customers,SILVER,VALID_EMAIL,Customer email must match expected email format,CRITICAL,1000,0,PASS,2026-09-29T23:50:41.199Z,2026-09-29T23:50:41.199Z


In [0]:
%sql
SELECT
    entity_name,
    source_system,
    business_key,
    load_enabled,
    cdc_enabled,
    load_sequence
FROM customer360_dev.control.pipeline_config
ORDER BY load_sequence;

entity_name,source_system,business_key,load_enabled,cdc_enabled,load_sequence
customers,CRM,customer_id,true,true,1
addresses,CRM,address_id,true,true,2
subscriptions,SUBSCRIPTION_PLATFORM,subscription_id,true,true,3
products,PRODUCT_CATALOG,product_id,true,true,4
orders,ORDER_MANAGEMENT,order_id,true,true,5
order_items,ORDER_MANAGEMENT,order_item_id,true,true,6
payments,PAYMENT_PLATFORM,payment_id,true,true,7
support_tickets,SUPPORT_PLATFORM,ticket_id,true,true,8
